In [0]:
# Configurações globais

VOLUMES = {
    "raw":  "/Volumes/workspace/olist/raw_data",
    "bronze": "/Volumes/workspace/olist/bronze",
    "silver": "/Volumes/workspace/olist/silver",
    "gold": "/Volumes/workspace/olist/gold",
    "dead_letter": "/Volumes/workspace/olist/dead_letter",
    "logs": "/Volumes/workspace/olist/logs"
}

TABELAS = [
    ("olist_orders_dataset.csv", "orders"),
    ("olist_order_items_dataset.csv", "order_items"),
    ("olist_customers_dataset.csv", "customers"),
    ("olist_products_dataset.csv", "products")
]

print("Configurações carregadas!")
print(f"Tabelas para processar: {[t[1] for t in TABELAS]}")

In [0]:
import json
from pyspark.sql.types import StructType

PRIMARY_KEYS = {
    "orders":      ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "customers":   ["customer_id"],
    "products":    ["product_id"]
}

NOT_NULL_COLS = {
    "orders":      ["order_id", "customer_id"],
    "order_items": ["order_id", "order_item_id", "product_id"],
    "customers":   ["customer_id", "customer_unique_id"],
    "products":    ["product_id"]
}

SCHEMA_PATH = "/Volumes/workspace/olist/bronze/_schemas"

def schema_existe(nome: str) -> bool:
    """Verifica se o schema já foi persistido no registry"""
    try:
        arquivos = dbutils.fs.ls(SCHEMA_PATH)
        return any(f.name == f"{nome}.json" for f in arquivos)
    except Exception:
        return False

def inferir_e_salvar_schema(nome: str, path_csv: str) -> StructType:
    schema_file = f"{SCHEMA_PATH}/{nome}.json"
    ja_existe = schema_existe(nome)
    
    if ja_existe:    
        try:
            # Schema já foi registrado - carrega e usa
            df_schema = spark.read.text(schema_file)
            schema_json = df_schema.collect()[0][0]
            schema = StructType.fromJson(json.loads(schema_json))
            print(f"  Schema carregado do registry: {nome}")
            return schema
        except Exception as e:
            # Schema existia mas falhou ao ler - não reinfere, exige intervenção
            raise Exception(
                f" [{nome}] CRITICO: Schema registry corrompido. "
                f" Intervenção manual necessária antes de continuar. Erro: {e}"
            )
    else:
        # Primeira execução - infere do CSV e persiste
        df_sample = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(path_csv, format="csv")
        
        schema = df_sample.schema
        dbutils.fs.put(schema_file, schema.json(), overwrite=True)
        print(f"  [{nome}] Schema inferido e salvo pela primeira vez ({len(schema.fields)} colunas)")
        return schema
    
# Inicializa o registry
dbutils.fs.mkdirs(SCHEMA_PATH)

print("Inicializando schema registry...")
SCHEMAS = {}
for arquivo, nome in TABELAS:
    path_csv = f"{VOLUMES['raw']}/{arquivo}"
    SCHEMAS[nome] = inferir_e_salvar_schema(nome, path_csv)

print(f"\nSchema registry pronto: {list(SCHEMAS.keys())}")

In [0]:
import uuid
from datetime import datetime, date
from pyspark.sql.types import (StructType, StructField, StringType, LongType, DoubleType, TimestampType)

LOG_PATH = "/Volumes/workspace/olist/bronze/_log"

# Schema explicito do log - nunca inferido
LOG_SCHEMA = StructType([
    StructField("execucao_id",        StringType(),    nullable=False),
    StructField("tabela",             StringType(),    nullable=False),
    StructField("status",             StringType(),    nullable=False),
    StructField("registros_lidos",    LongType(),      nullable=True),
    StructField("registros_gravados", LongType(),      nullable=True),
    StructField("divergencia_pct",    DoubleType(),    nullable=True),
    StructField("tempo_segundos",     DoubleType(),    nullable=True),
    StructField("mensagem",           StringType(),    nullable=True),
    StructField("timestamp",          TimestampType(), nullable=False)
])

def gravar_log(execucao_id: str, tabela: str, status: str,
               registros_lidos: int = None, registros_gravados: int = None,
               divergencia_pct: float = None, tempo_segundos: float = None,
               mensagem: str = None):
    
    linha = [{
        "execucao_id":        execucao_id,
        "tabela":             tabela,
        "status":             status,
        "registros_lidos":    registros_lidos,
        "registros_gravados": registros_gravados,
        "divergencia_pct":    divergencia_pct,
        "tempo_segundos":     tempo_segundos,
        "mensagem":           mensagem,
        "timestamp":          datetime.now()
    }]
    
    df_log = spark.createDataFrame(linha, schema=LOG_SCHEMA)
    
    df_log.write \
        .format("delta") \
        .mode("append") \
        .save(LOG_PATH)
        
# Gera ID único para essa execução — agrupa todos os logs da mesma rodada
EXECUCAO_ID = str(uuid.uuid4())

print(f"Log inicializado!")
print(f"Execucao ID: {EXECUCAO_ID}")
print(f"Log path: {LOG_PATH}")

# Teste — grava uma linha de log para validar
gravar_log(
    execucao_id    = EXECUCAO_ID,
    tabela         = "PIPELINE",
    status         = "STARTED",
    mensagem       = "Inicialização do pipeline de Bronze"
)

# Confirma que gravou
spark.read.format("delta").load(LOG_PATH).show(truncate=False)

In [0]:
from pyspark.sql import DataFrame

def validar_qualidade(df: DataFrame, nome: str, schema_esperado) -> dict:
    """
    Valida a qualidade do DataFrame antes de gravar no Bronze.
    Retorna dicionário com status e lista de problemas encontrados.
    """
    problemas = []
    warnings = []
    
    # 1. Valida colunas obrigatórias
    colunas_esperadas = set([f.name for f in schema_esperado.fields])
    colunas_recebidas = set(df.columns)
    colunas_faltando = colunas_esperadas - colunas_recebidas
    
    if colunas_faltando:
        problemas.append(f"Colunas faltando: {colunas_faltando}")
    
    colunas_extras = colunas_recebidas - colunas_esperadas
    
    if colunas_extras:
        warnings.append(f"Colunas extras inesperadas: {colunas_extras}")

    # 2 Valida nulos em colunas críticas
    for coluna in NOT_NULL_COLS.get(nome, []):
        if coluna in df.columns:
            nulos = df.filter(df[coluna].isNull()).count()
            if nulos > 0:
                problemas.append(f"Coluna '{coluna}' tem {nulos} valores nulos")
    
    # 3. Valida duplicatas em chaves primárias
    pks = PRIMARY_KEYS.get(nome, [])
    if pks and all(pk in df.columns for pk in pks):
        total   = df.count()
        distintos   = df.select(pks).distinct().count()
        duplicatas  = total - distintos
        if duplicatas > 0:
            warnings.append(f"Chave primária {pks} tem {duplicatas} duplicatas")

    # Resultado final
    valido = len(problemas) == 0

    return {

        "valido": valido,
        "problemas": problemas,
        "warnings": warnings
    }

print("DataQualityValidator carregado!")

#Teste rápido - valida a tabela orders

# Teste rápido — valida a tabela orders
df_teste = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOLUMES['raw']}/olist_orders_dataset.csv", format="csv")

resultado = validar_qualidade(df_teste, "orders", SCHEMAS["orders"])

print(f"\nResultado da validação — orders:")
print(f"  Válido:    {resultado['valido']}")
print(f"  Problemas: {resultado['problemas']}")
print(f"  Warnings:  {resultado['warnings']}")

In [0]:
import time
from datetime import datetime, date
from pyspark.sql import functions as F

def gravar_dead_letter(df: DataFrame, nome: str, motivo: str, execucao_id: str):
    """
    Grava registros problemáticos na Dead Letter Queue
    com metadados do motivo da falha
    """

    df_dl = df \
        .withColumn("_dl_motivo",       F.lit(motivo)) \
        .withColumn("_dl_execucao_id",  F.lit(execucao_id)) \
        .withColumn("_dl_tabela",       F.lit(nome)) \
        .withColumn("_dl_timestamp",    F.now())

    df_dl.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .save(f"{VOLUMES['dead_letter']}/{nome}")
    
    print(f" [{nome}] {df_dl.count()} registros enviados para dead_letter")

def processar_tabela(arquivo: str, nome: str, execucao_id: str):
    """
    Orquestra o processamento completo de uma tabela:
    leitura -> validação -> bronze ou dead_letter -> log
    """

    inicio = time.time()
    path_csv = f"{VOLUMES['raw']}/{arquivo}"
    print(f"\n{'='*50}")
    print(f"Processando: {nome}")
    print(f"{'='*50}")

    # ETAPA 1 - Leitura com schema do registry
    try:
        df = spark.read \
            .option("header", "true") \
            .schema(SCHEMAS[nome]) \
            .load(path_csv, format="csv")

        registros_lidos = df.count()
        print(f" [{nome}] Lidos: {registros_lidos} registros")

    except Exception as e:
        tempo = round(time.time() - inicio, 2)
        gravar_log(
            execucao_id             =  execucao_id,
            tabela                  =  nome,
            status                  =  "FAILED",
            tempo_segundos          = tempo,
            mensagem                =  f"Erro na leitura do CSV: {str(e)}"  
        )
        print(f"    [{nome}]    FALHA na leitura:   {e}")
        return
    
    # ETAPA 2 - Validação de qualidade
    resultado = validar_qualidade(df, nome, SCHEMAS[nome])

    if resultado["warnings"]:
        for w in resultado["warnings"]:
            print(f"    [{nome}]    WARNING: {w}")

    if not resultado["valido"]:
        motivo = " | ".join(resultado["problemas"])
        gravar_dead_letter(df, nome, motivo, execucao_id)
        tempo = round(time.time() - inicio, 2)
        gravar_log(
            execucao_id             =  execucao_id,
            tabela                  =  nome,
            status                  =  "FAILED",
            registros_lidos         =  registros_lidos,
            tempo_segundos          = tempo,
            mensagem                =  f"Falha na validação: {motivo}"  
        )

        print(f"    [{nome}]    FALHA na validação - dados enviados para dead_letter")
        return
    
    # ETAPA 3 - Adiciona colunas de auditoria
    df_bronze = df \
        .withColumn("_ingest_timestamp", F.now()) \
        .withColumn("_source_filename", F.col("_metadata.file_path")) \
        .withColumn("_processing_date", F.lit(str(date.today()))) \
        .withColumn("_execucao_id",     F.lit(execucao_id))

    # ETAPA 4 - Grava na tabela bronze
    try:
        df_bronze.write \
            .format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .save(f"{VOLUMES['bronze']}/{nome}")

        registros_gravados = spark.read \
            .format("delta") \
             .load(f"{VOLUMES['bronze']}/{nome}") \
             .count()

        print(f"    [{nome}] Gravados: {registros_gravados} registros")

    except Exception as e:
        tempo = round(time.time() - inicio, 2)
        gravar_log(
            execucao_id             =  execucao_id,
            tabela                  =  nome,
            status                  =  "FAILED",
            registros_lidos         =  registros_lidos,
            tempo_segundos          = tempo,
            mensagem                =  f"Erro na gravação Bronze: {str(e)}"  
        )
        print(f"    [{nome}]    FALHA na gravação: {e}")
        return

    # ETAPA 5 - Validação de divergência
    divergencia_pct = abs(registros_lidos - registros_gravados) / registros_lidos * 100
    tempo = round(time.time() - inicio, 2)
    
    if divergencia_pct == 0:
        status = "SUCCESS"
        mensagem = f"Processado com sucesso em {tempo}s"
    elif divergencia_pct <= 1:
        status = "WARNING"
        mensagem = f"Divergência aceitavel: {divergencia_pct:.2f}%"
    elif divergencia_pct <= 5:
        status  = "WARNING"
        mensagem = f"Divergência moderada: {divergencia_pct:.2f}% - monitorar"
    else:
        status = "FAILED"
        mensagem = f"Divergência crítica: {divergencia_pct:.2f}% - investigar"
        gravar_dead_letter(df, nome, mensagem, execucao_id)

    gravar_log(
        execucao_id             =  execucao_id,
        tabela                  =  nome,
        status                  =  status,
        registros_lidos         =  registros_lidos,
        registros_gravados      =  registros_gravados,
        divergencia_pct         =  divergencia_pct,
        tempo_segundos          =  tempo,
        mensagem                =  mensagem  
    )
    print(f"  [{nome}] Status: {status} | Divergência: {divergencia_pct:.2f}% | Tempo: {tempo}s")


print("Orquestrador carregado!")



In [0]:
from pyspark.sql import functions as F
from datetime import datetime, date

# Gera novo ID de execução
EXECUCAO_ID = str(uuid.uuid4())

print(f"Iniciando pipeline Bronze — Execução: {EXECUCAO_ID}")
print(f"Timestamp: {datetime.now()}")

# Registra início no log
gravar_log(
    execucao_id = EXECUCAO_ID,
    tabela      = "PIPELINE",
    status      = "STARTED",
    mensagem    = f"Início do pipeline Bronze — {len(TABELAS)} tabelas para processar"
)

# Processa cada tabela de forma isolada
resultados = {"SUCCESS": [], "WARNING": [], "FAILED": []}

for arquivo, nome in TABELAS:
    try:
        processar_tabela(arquivo, nome, EXECUCAO_ID)
    except Exception as e:
        print(f"  [{nome}] ERRO INESPERADO: {e}")
        gravar_log(
            execucao_id = EXECUCAO_ID,
            tabela      = nome,
            status      = "FAILED",
            mensagem    = f"Erro inesperado: {str(e)}"
        )

# Lê o log dessa execução para resumo final
df_resumo = spark.read \
    .format("delta") \
    .load(LOG_PATH) \
    .filter(F.col("execucao_id") == EXECUCAO_ID) \
    .filter(F.col("tabela") != "PIPELINE")

print(f"\n{'='*50}")
print(f"RESUMO DA EXECUÇÃO — {EXECUCAO_ID}")
print(f"{'='*50}")

df_resumo.select(
    "tabela",
    "status",
    "registros_lidos",
    "registros_gravados",
    "divergencia_pct",
    "tempo_segundos",
    "mensagem"
).show(truncate=False)

# Registra fim no log
total     = df_resumo.count()
sucessos  = df_resumo.filter(F.col("status") == "SUCCESS").count()
warnings  = df_resumo.filter(F.col("status") == "WARNING").count()
falhas    = df_resumo.filter(F.col("status") == "FAILED").count()

gravar_log(
    execucao_id = EXECUCAO_ID,
    tabela      = "PIPELINE",
    status      = "FINISHED" if falhas == 0 else "FINISHED_WITH_ERRORS",
    mensagem    = f"Concluído — SUCCESS: {sucessos} | WARNING: {warnings} | FAILED: {falhas}"
)

print(f"\nSUCCESS: {sucessos} | WARNING: {warnings} | FAILED: {falhas}")
print(f"Pipeline finalizado!")